In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast

In [2]:
df=pd.read_csv(r"C:\Users\SAMA\Documents\Olist\olist_orders_dataset.csv")

In [3]:
df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,10/2/2017 10:56,10/2/2017 11:07,10/4/2017 19:55,10/10/2017 21:25,10/18/2017 0:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,7/24/2018 20:41,7/26/2018 3:24,7/26/2018 14:31,8/7/2018 15:27,8/13/2018 0:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,8/8/2018 8:38,8/8/2018 8:55,8/8/2018 13:50,8/17/2018 18:06,9/4/2018 0:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,11/18/2017 19:28,11/18/2017 19:45,11/22/2017 13:39,12/2/2017 0:28,12/15/2017 0:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2/13/2018 21:18,2/13/2018 22:20,2/14/2018 19:46,2/16/2018 18:17,2/26/2018 0:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,3/9/2017 9:54,3/9/2017 9:54,3/10/2017 11:18,3/17/2017 15:08,3/28/2017 0:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2/6/2018 12:58,2/6/2018 13:10,2/7/2018 23:22,2/28/2018 17:37,3/2/2018 0:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,8/27/2017 14:46,8/27/2017 15:04,8/28/2017 20:52,9/21/2017 11:24,9/27/2017 0:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,1/8/2018 21:28,1/8/2018 21:36,1/12/2018 15:35,1/25/2018 23:32,2/15/2018 0:00


In [3]:
df_check=df[df['order_approved_at'].isna()].copy()

In [4]:
df_check['order_status'].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [5]:
df_cleaned=df.dropna(subset='order_approved_at').copy()

In [6]:
df_cleaned.info()

<class 'pandas.DataFrame'>
Index: 99281 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99281 non-null  str  
 1   customer_id                    99281 non-null  str  
 2   order_status                   99281 non-null  str  
 3   order_purchase_timestamp       99281 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97644 non-null  str  
 6   order_delivered_customer_date  96462 non-null  str  
 7   order_estimated_delivery_date  99281 non-null  str  
dtypes: str(8)
memory usage: 20.6 MB


In [7]:
df_cleaned['order_approved_at']=pd.to_datetime(df['order_approved_at'])
df_cleaned['order_purchase_timestamp']=pd.to_datetime(df['order_purchase_timestamp'])
df_cleaned['order_delivered_carrier_date']=pd.to_datetime(df['order_delivered_carrier_date'])
df_cleaned['order_delivered_customer_date']=pd.to_datetime(df['order_delivered_customer_date'])
df_cleaned['order_estimated_delivery_date']=pd.to_datetime(df['order_estimated_delivery_date'])

In [8]:
df_cleaned['approval_time_hours'] = (
    df_cleaned['order_approved_at'] -
    df_cleaned['order_purchase_timestamp']
).dt.total_seconds() / 3600

In [9]:
df_cleaned['approval_time_hours'].describe()

count    99281.000000
mean        10.419907
std         26.037777
min          0.000000
25%          0.216667
50%          0.350000
75%         14.583333
max       4509.183333
Name: approval_time_hours, dtype: float64

In [10]:
df_cleaned['approval_time_hours'].quantile([0.90, 0.95, 0.99])

0.90    34.666667
0.95    48.466667
0.99    90.170000
Name: approval_time_hours, dtype: float64

In [11]:
df_cleaned['delivery_time_days'] = (
    df_cleaned['order_delivered_customer_date']
    - df_cleaned['order_purchase_timestamp']
).dt.days

df_cleaned['delivery_delay_days'] = (
    df_cleaned['order_delivered_customer_date']
    - df_cleaned['order_estimated_delivery_date']
).dt.days

In [12]:
df_cleaned[df_cleaned['delivery_delay_days']>0]['delivery_delay_days'].describe()

count    6535.000000
mean       10.620352
std        14.643844
min         1.000000
25%         3.000000
50%         7.000000
75%        13.000000
max       188.000000
Name: delivery_delay_days, dtype: float64

In [13]:
df_cleaned['delivery_delay_days'].quantile([0.90,0.95,0.99])

0.90    -2.0
0.95     3.0
0.99    18.0
Name: delivery_delay_days, dtype: float64

In [14]:
(df_cleaned['delivery_delay_days'] > 18).sum()

np.int64(961)

In [15]:
(
    df_cleaned['order_delivered_carrier_date']
    - df_cleaned['order_approved_at']
).describe()

count                     97644
mean     2 days 19:19:19.008848
std      3 days 13:11:09.266568
min         -172 days +18:45:00
25%             0 days 21:01:00
50%             1 days 19:39:00
75%             3 days 13:56:00
max           125 days 18:18:00
dtype: object

In [16]:
df_cleaned[['order_approved_at','order_delivered_carrier_date']].sample(10)

,order_approved_at,order_delivered_carrier_date
62063,2018-06-25 09:35:00,2018-06-28 14:42:00
55811,2017-05-16 03:22:00,2017-05-16 15:17:00
86497,2017-03-24 06:35:00,2017-03-24 15:05:00
7881,2017-05-17 18:25:00,2017-05-19 14:49:00
18944,2017-12-05 17:12:00,2017-12-08 01:39:00
34060,2017-07-29 01:25:00,2017-07-31 16:41:00
14545,2018-07-05 16:26:00,2018-07-05 14:13:00
49022,2018-07-24 16:05:00,2018-07-25 14:21:00
7756,2017-06-29 20:55:00,2017-06-30 14:10:00
13126,2017-04-30 20:31:00,2017-05-04 06:53:00


In [17]:
bad_orders = (
    df_cleaned['order_delivered_carrier_date']
    - df_cleaned['order_approved_at']
).dt.days

bad_orders[bad_orders < 0].describe()

count    1358.000000
mean       -1.760677
std         4.798613
min      -172.000000
25%        -2.000000
50%        -1.000000
75%        -1.000000
max        -1.000000
dtype: float64

In [18]:
carrier_after_approval = (
    df_cleaned['order_delivered_carrier_date']
    - df_cleaned['order_approved_at']
).dt.total_seconds() / 3600

(carrier_after_approval < -24).sum()

np.int64(457)

In [19]:
df_cleaned = df_cleaned[carrier_after_approval >= -24]

In [20]:
df_cleaned.info()

<class 'pandas.DataFrame'>
Index: 97187 entries, 0 to 99440
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       97187 non-null  str           
 1   customer_id                    97187 non-null  str           
 2   order_status                   97187 non-null  str           
 3   order_purchase_timestamp       97187 non-null  datetime64[us]
 4   order_approved_at              97187 non-null  datetime64[us]
 5   order_delivered_carrier_date   97187 non-null  datetime64[us]
 6   order_delivered_customer_date  96009 non-null  datetime64[us]
 7   order_estimated_delivery_date  97187 non-null  datetime64[us]
 8   approval_time_hours            97187 non-null  float64       
 9   delivery_time_days             96009 non-null  float64       
 10  delivery_delay_days            96009 non-null  float64       
dtypes: datetime64[us](5), float64(3

In [21]:
df_cleaned.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   0
order_delivered_carrier_date        0
order_delivered_customer_date    1178
order_estimated_delivery_date       0
approval_time_hours                 0
delivery_time_days               1178
delivery_delay_days              1178
dtype: int64

In [22]:
df_cleaned[df_cleaned['order_delivered_customer_date'].isna()]['order_status'].value_counts()

order_status
shipped      1102
canceled       69
delivered       7
Name: count, dtype: int64

In [23]:
df_cleaned.to_csv('orders_clean.csv')